In [1]:
from google.colab import drive
drive.mount('/content/drive')

# ! pip install numpyro
# # ! pip install jax_cosmo
# ! pip install arviz
! pip install jaxopt
! pip install corner
! pip install emcee

path = '/content/drive/MyDrive/SchwarMAX-MCMC/'

import sys
sys.path.append(path)

from likelihoods import *
from utils import *

import jax
import jax.numpy as jnp
import jax.numpy.linalg as jnn
import pandas as pd
import numpy as np
import scipy as sp
import pickle

import emcee
import corner
import matplotlib.pyplot as plt

from constants import EPSILON

def get_dict_data(path):
    df_ic = pd.read_csv(path + 'mock_initial_conditions_xyz.csv')
    df_ic = df_ic[np.sqrt(df_ic['x']**2 + df_ic['y']**2) < 15.0]
    df_ic = df_ic[np.fabs(df_ic['z']) < 4.0]

    n_particles =  20_000
    print(n_particles)
    np.random.seed(42)
    index = np.random.choice(len(df_ic['x']), size=n_particles, replace=False)
    df_ic = df_ic.iloc[index]
    # w0 = jnp.array([df_ic['x'], df_ic['y'], df_ic['z'], df_ic['vx'], df_ic['vy'], df_ic['vz']]).T
    w0 = jnp.array(df_ic[['x','y','z']].to_numpy())

    with open(path + 'mock_SCM_disc_XY_withRot.pkl', 'rb') as f:
        bin_dict = pickle.load(f)

    # voronoi binning mapping and data
    num_per_bin = jnp.array(bin_dict['num_per_bin'])
    total_bins = jnp.array(bin_dict['total_bins'])
    bin_mapping = jnp.array(bin_dict['bin_mapping'])
    surface_density = jnp.array(bin_dict['surface_density'])
    V_data = jnp.array(bin_dict['V_mean'])
    sigma_data = jnp.array(bin_dict['V_sigma'])
    h1_data = jnp.array(bin_dict['h1'])
    h2_data = jnp.array(bin_dict['h2'])
    h3_data = jnp.array(bin_dict['h3'])
    h4_data = jnp.array(bin_dict['h4'])
    v0 = jnp.array(bin_dict['v0'])
    s = jnp.array(bin_dict['s'])
    alpha, beta, gamma = bin_dict['orientation']

    V_data_err = jnp.where(0.1 * jnp.fabs(V_data) < 10, 10, 0.1 * V_data)
    sigma_data_err = jnp.where(0.1 * jnp.fabs(sigma_data) < 5, 5, 0.1 * sigma_data)
    h1_data_err = jnp.where(0.1 * jnp.fabs(h1_data) < 0.03, 0.03, 0.1 * jnp.fabs(h1_data))
    h2_data_err = jnp.where(0.1 * jnp.fabs(h2_data) < 0.03, 0.03, 0.1 * jnp.fabs(h2_data))
    h3_data_err = jnp.where(0.1 * jnp.fabs(h3_data) < 0.03, 0.03, 0.1 * jnp.fabs(h3_data))
    h4_data_err = jnp.where(0.1 * jnp.fabs(h4_data) < 0.03, 0.03, 0.1 * jnp.fabs(h4_data))

    # df_Rzphi_data = pd.read_csv(path + 'mock_axisymmetric_disc_Rzphi.csv')
    # Rzphi_density_data = jnp.array(df_Rzphi_data['mass'].to_numpy()).astype(jnp.float32)
    with open(path + 'mock_axisymmetric_disc_Rzphi.pkl', 'rb') as f:
        Rzphi_density_data = pickle.load(f)

    R_grid, z_grid, phi_grid = Rzphi_density_data['R_grid'], Rzphi_density_data['z_grid'], Rzphi_density_data['phi_grid']
    dR = np.unique(R_grid)[1] - np.unique(R_grid)[0]
    dz = np.unique(z_grid)[1] - np.unique(z_grid)[0]
    dphi = np.unique(phi_grid)[1] - np.unique(phi_grid)[0]
    sample_for_integration = Rzphi_density_data['sample_for_integration']

    from scipy.stats import qmc
    X_regular_grid, Y_regular_grid = bin_dict['X_regular_grid'], bin_dict['Y_regular_grid']
    dX = jnp.unique(X_regular_grid)[1] - jnp.unique(X_regular_grid)[0]
    dY = jnp.unique(Y_regular_grid)[1] - jnp.unique(Y_regular_grid)[0]
    sampler = qmc.Sobol(d=3, scramble=False)
    sample = sampler.random_base2(m=10)


    dict_data = {
        'w0': w0,
        'v0': v0,
        's': s,

        # 'Rzphi_density_data': Rzphi_density_data,
        'XY_density_data': surface_density,
        'V_data': V_data,
        'V_data_err': V_data_err,
        'sigma_data': sigma_data,
        'sigma_data_err': sigma_data_err,
        'h1_data': h1_data,
        'h1_data_err': h1_data_err,
        'h2_data': h2_data,
        'h2_data_err': h2_data_err,
        'h3_data': h3_data,
        'h3_data_err': h3_data_err,
        'h4_data': h4_data,
        'h4_data_err': h4_data_err,
        'num_per_bin': num_per_bin,
        'bin_mapping': bin_mapping,
        'total_bins': total_bins.item(),

        'R_grid': R_grid,
        'z_grid': z_grid,
        'phi_grid': phi_grid,
        'dR': dR,
        'dz': dz,
        'dphi': dphi,
        'sample_for_integration': sample_for_integration,

        'X_regular_grid': X_regular_grid,
        'Y_regular_grid': Y_regular_grid,
        'dX': dX,
        'dY': dY,
        'sample_for_integration_XY': sample,
    }

    return dict_data

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.4/172.4 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 5.2 MB/s eta 0:00:00


In [ ]:
def mapping_scale_uniform_to_norm(q, angle, min=0.5, max=1.5):
    """
    Computes the scale length/height from the direction vector components. Uniform [0.5, 1.5].
    """
    # r = jnp.sqrt(dirx**2 + diry**2)
    # q = 1 - jnp.exp(-r**2/2)
    # q = (max-min)*q + min

    q = (q-min) / (max-min)
    r = jnp.sqrt(-2*jnp.log(1-q))

    return r * jnp.cos(angle), r * jnp.sin(angle)



dict_data = get_dict_data(path)

with open(path + '/mock_axisymmetric_disc_potential_params.pkl', 'rb') as f:
    gt_params = pickle.load(f)

N = 20
i = 10

# alpha = 30 * np.pi/180
# beta = 20 * np.pi/180
# gamma = 0 * np.pi/180

# x_alpha, y_alpha = mapping_scale_uniform_to_norm(jnp.log10(gt_params['halo_params']['scaleRadius']).item(), alpha, min=0.5, max=1.5)
# x_beta, y_beta = mapping_scale_uniform_to_norm(jnp.log10(gt_params['disc_params']['scaleRadius']).item(), beta, min=0., max=1.0)
# x_gamma, y_gamma = mapping_scale_uniform_to_norm(jnp.log10(gt_params['disc_params']['scaleHeight']).item(), gamma, min=-1.0, max=0.)
# ground_truth = [gt_params['halo_params']['logM'].item() + (i-N/2)/10,
#                 gt_params['disc_params']['logM'].item(),
#                 x_alpha,
#                 y_alpha,
#                 x_beta,
#                 y_beta,
#                 x_gamma,
#                 y_gamma,
# ]

# logL = logl(ground_truth, dict_data, dict_data['total_bins'])
# print(logL)

alpha = 30 * np.pi/180
beta = 20 * np.pi/180
gamma = 0 * np.pi/180
ground_truth = [gt_params['halo_params']['logM'].item() + (i-N/2)/10,
                gt_params['disc_params']['logM'].item(),
                jnp.log10(gt_params['halo_params']['scaleRadius']).item(),
                jnp.log10(gt_params['disc_params']['scaleRadius']).item(),
                jnp.log10(gt_params['disc_params']['scaleHeight']).item(),
                alpha,
                beta,
                gamma,
]
logL = logl_angular_input(ground_truth, dict_data, dict_data['total_bins'])
print(logL)

20000
-8.457225


In [ ]:
import emcee
import numpy as np

@jax.jit
def log_gaussian(x, mu, sig):
    return -0.5 * jnp.log(2 * jnp.pi * sig**2) - (x - mu)**2 / (2 * sig**2)


param_names = ['logM_halo','logM_disk', 'x_alpha', 'y_alpha', 'x_beta', 'y_beta', 'x_gamma', 'y_gamma']

PATH_DATA = f'/content/drive/MyDrive/SchwarMAX'

dict_data = get_dict_data(path)


with open(path + '/mock_axisymmetric_disc_potential_params.pkl', 'rb') as f:
    gt_params = pickle.load(f)

ground_truth = {
        'logM_halo': gt_params['halo_params']['logM'].item(),
        'logM_disk': gt_params['disc_params']['logM'].item(),
}

prior_uniform_low = [gt_params['halo_params']['logM'].item() - 3, gt_params['disc_params']['logM'].item() - 3]
prior_uniform_high = [gt_params['halo_params']['logM'].item() + 3, gt_params['disc_params']['logM'].item() + 3]

def log_prior(params):
  lp = 0
  if (params[0]<prior_uniform_low[0]) & (params[0]>prior_uniform_high[0]) & (params[1]<prior_uniform_low[1]) & (params[1]>prior_uniform_high[1]):
    lp = -jnp.inf

  for i in range (2, 8):
    lp+= log_gaussian(params[i], 0, 1)
  return lp

def log_prob(theta):
    params = theta
    lp = log_prior(params)
    if not np.isfinite(lp):
        return -np.inf
    ll = float(logl(params, dict_data, dict_data['total_bins']))  # convert from JAX array
    if not np.isfinite(ll):
        return -np.inf
    return lp + ll


ndim = 8
nwalkers = 16  # must be >= 2 * ndim

# Initialize walkers around ground truth
# p0 = np.array([ground_truth[k] for k in param_names])
p0 = np.array([12, 10.2, 0, 0, 0, 0, 0, 0])
initial_pos = p0 + 1e-1 * np.random.randn(nwalkers, ndim)

sampler = emcee.EnsembleSampler(nwalkers, ndim, log_prob)
sampler.run_mcmc(initial_pos, 500, progress=True)

samples = sampler.get_chain(discard=100, flat=True)

import pandas as pd
pd.DataFrame(samples, columns=param_names).to_csv('/content/drive/MyDrive/SchwarMAX-MCMC/test_posterior_0206.csv', index=False)

samples_raw = sampler.get_chain(discard=0, flat=False)
with open('/content/drive/MyDrive/SchwarMAX-MCMC/test_posterior_WholeChain_0206.pkl', 'wb') as f:
  pickle.dump(samples_raw, f)

20000


100%|██████████| 500/500 [22:21:14<00:00, 160.95s/it]


In [ ]:
import emcee
import numpy as np

@jax.jit
def log_gaussian(x, mu, sig):
    return -0.5 * jnp.log(2 * jnp.pi * sig**2) - (x - mu)**2 / (2 * sig**2)


param_names = ['logM_halo','logM_disk', 'logRs_halo', 'logRs_disk', 'logHs_disk', 'alpha', 'beta', 'gamma']

# PATH_DATA = f'/content/drive/MyDrive/SchwarMAX'

dict_data = get_dict_data(path)


with open(path + '/mock_axisymmetric_disc_potential_params.pkl', 'rb') as f:
    gt_params = pickle.load(f)

ground_truth = {
        'logM_halo': gt_params['halo_params']['logM'].item(),
        'logM_disk': gt_params['disc_params']['logM'].item(),
        'logRs_halo': jnp.log10(gt_params['halo_params']['scaleRadius']).item(),
        'logRs_disk': jnp.log10(gt_params['disc_params']['scaleRadius']).item(),
        'logHs_disk': jnp.log10(gt_params['disc_params']['scaleHeight']).item(),
}

prior_uniform_low =  [ground_truth['logM_halo'] - 3,
                      ground_truth['logM_disk'] - 3,
                      ground_truth['logRs_halo']- 1,
                      ground_truth['logRs_disk']- 1,
                      ground_truth['logHs_disk']- 1,
                      -jnp.pi,
                      -jnp.pi,
                      -jnp.pi,
                      ]
prior_uniform_high = [ground_truth['logM_halo'] + 3,
                      ground_truth['logM_disk'] + 3,
                      ground_truth['logRs_halo']+ 1,
                      ground_truth['logRs_disk']+ 1,
                      ground_truth['logHs_disk']+ 1,
                      jnp.pi,
                      jnp.pi,
                      jnp.pi,
                      ]

def log_prior(params):
  lp = 0
  for i in range (0, 8):
      if (params[i]<prior_uniform_low[i]) & (params[i]>prior_uniform_high[i]):
        lp+= -jnp.inf
  return lp

def log_prob(theta):
    params = theta
    lp = log_prior(params)
    if not np.isfinite(lp):
        return -np.inf
    ll = float(logl_angular_input(params, dict_data, dict_data['total_bins']))  # convert from JAX array
    if not np.isfinite(ll):
        return -np.inf
    return lp + ll


ndim = 8
nwalkers = 16  # must be >= 2 * ndim

# Initialize walkers around ground truth
# p0 = np.array([ground_truth[k] for k in param_names])
p0 = np.array([12, 10.2, 1, 0.5, 0, 0, 0, 0])
# initial_pos = p0 + 1e-1 * np.random.randn(nwalkers, ndim)
initial_pos = p0 + np.random.uniform(-0.8, 0.8, (nwalkers, ndim))

sampler = emcee.EnsembleSampler(nwalkers, ndim, log_prob)
sampler.run_mcmc(initial_pos, 300, progress=True)

samples = sampler.get_chain(discard=100, flat=True)

import pandas as pd
pd.DataFrame(samples, columns=param_names).to_csv('/content/drive/MyDrive/SchwarMAX-MCMC/test_posterior_0210.csv', index=False)

samples_raw = sampler.get_chain(discard=0, flat=False)
with open('/content/drive/MyDrive/SchwarMAX-MCMC/test_posterior_WholeChain_0210.pkl', 'wb') as f:
  pickle.dump(samples_raw, f)

20000


100%|██████████| 300/300 [11:33:18<00:00, 138.66s/it]


In [ ]:


dict_data = get_dict_data(path)

ndim = 5
num_Vbin = dict_data['total_bins']
nlive = 500
use_cpu = False
if use_cpu:
    dns_results = cpu_dynesty_fit(dict_data, dynesty_logl, prior_transform, ndim, num_Vbin, nlive=nlive)
else:
    dns_results = gpu_dynesty_fit(dict_data, dynesty_logl, prior_transform, ndim, num_Vbin, nlive=nlive)
with open(f'{path}/dict_results_test.pkl', 'wb') as f:
    pickle.dump(dns_results, f)

with open(path + '/mock_axisymmetric_disc_potential_params.pkl', 'rb') as f:
    gt_params = pickle.load(f)

ground_truth = {
        'logM_halo': gt_params['halo_params']['logM'].item(),
        'logRs_halo': jnp.log10(gt_params['halo_params']['scaleRadius']).item(),
        'logM_disk': gt_params['disc_params']['logM'].item(),
        'logRs_disk': jnp.log10(gt_params['disc_params']['scaleRadius']).item(),
        'logHs_disk': jnp.log10(gt_params['disc_params']['scaleHeight']).item(),
}

# Plot and Save corner plot
labels = ['logM', 'logRs', 'logm', 'logrs', 'loghs']
figure = corner.corner(dns_results['samps'],
            labels=labels,
            color='blue',
            quantiles=[0.16, 0.5, 0.84],
            show_titles=True,
            title_kwargs={"fontsize": 16},
            truths=[ground_truth['logM_halo'], ground_truth['logRs_halo'], ground_truth['logM_disk'], ground_truth['logRs_disk'], ground_truth['logHs_disk']],
            truth_color='red',
            )
figure.savefig(f'{path}/corner_plot.pdf')
plt.close(figure)


20000


0it [00:00, ?it/s]

# Grid search

In [ ]:
import numpy as np
from tqdm import tqdm

@jax.jit
def log_gaussian(x, mu, sig):
    return -0.5 * jnp.log(2 * jnp.pi * sig**2) - (x - mu)**2 / (2 * sig**2)


param_names = ['logM_halo','logM_disk', 'x_alpha', 'y_alpha', 'x_beta', 'y_beta', 'x_gamma', 'y_gamma']

PATH_DATA = f'/content/drive/MyDrive/SchwarMAX'

dict_data = get_dict_data(path)


with open(path + '/mock_axisymmetric_disc_potential_params.pkl', 'rb') as f:
    gt_params = pickle.load(f)

ground_truth = {
        'logM_halo': gt_params['halo_params']['logM'].item(),
        'logM_disk': gt_params['disc_params']['logM'].item(),
}

prior_uniform_low = [gt_params['halo_params']['logM'].item() - 3, gt_params['disc_params']['logM'].item() - 3]
prior_uniform_high = [gt_params['halo_params']['logM'].item() + 3, gt_params['disc_params']['logM'].item() + 3]

def log_prior(params):
  lp = 0
  if (params[0]<prior_uniform_low[0]) & (params[0]>prior_uniform_high[0]) & (params[1]<prior_uniform_low[1]) & (params[1]>prior_uniform_high[1]):
    lp = -jnp.inf

  for i in range (2, 8):
    lp+= log_gaussian(params[i], 0, 1)
  return lp

def log_prob(theta):
    params = theta
    lp = log_prior(params)
    if not np.isfinite(lp):
        return -np.inf
    ll = float(logl(params, dict_data, dict_data['total_bins']))  # convert from JAX array
    if not np.isfinite(ll):
        return -np.inf
    return lp + ll


n_grid = 1024
param_grid = pd.read_csv(path + '/quasi_random_samples_8D_unity.csv').to_numpy()
index = np.random.choice(len(param_grid), size=n_grid, replace=False)
param_grid = param_grid[index]
param_grid[:, 0] = (param_grid[:, 0] - 0.5) * 6 + ground_truth['logM_halo']
param_grid[:, 1] = (param_grid[:, 1] - 0.5) * 6 + ground_truth['logM_disk']

param_grid[:, 2] = (param_grid[:, 2] - 0.5) * 4
param_grid[:, 3] = (param_grid[:, 3] - 0.5) * 4
param_grid[:, 4] = (param_grid[:, 4] - 0.5) * 4
param_grid[:, 5] = (param_grid[:, 5] - 0.5) * 4
param_grid[:, 6] = (param_grid[:, 6] - 0.5) * 4
param_grid[:, 7] = (param_grid[:, 7] - 0.5) * 4

log_prob_grid = []
for i in tqdm(range(n_grid)):
  log_prob_grid.append(log_prob(param_grid[i]))
log_prob_grid = np.array(log_prob_grid)

pd.DataFrame({
    'logM_halo': param_grid[:, 0],
    'logM_disk': param_grid[:, 1],
    'x_alpha': param_grid[:, 2],
    'y_alpha': param_grid[:, 3],
    'x_beta': param_grid[:, 4],
    'y_beta': param_grid[:, 5],
    'x_gamma': param_grid[:, 6],
    'y_gamma': param_grid[:, 7],
    'log_prob': log_prob_grid,
}).to_csv(path + '/grid_search_result_0208.csv', index=False)

20000


100%|██████████| 1024/1024 [3:08:24<00:00, 11.04s/it]


In [ ]:
import numpy as np
from tqdm import tqdm

@jax.jit
def log_gaussian(x, mu, sig):
    return -0.5 * jnp.log(2 * jnp.pi * sig**2) - (x - mu)**2 / (2 * sig**2)


param_names = ['logM_halo','logM_disk', 'x_alpha', 'y_alpha', 'x_beta', 'y_beta', 'x_gamma', 'y_gamma']

PATH_DATA = f'/content/drive/MyDrive/SchwarMAX'

dict_data = get_dict_data(path)


with open(path + '/mock_axisymmetric_disc_potential_params.pkl', 'rb') as f:
    gt_params = pickle.load(f)

ground_truth = {
        'logM_halo': gt_params['halo_params']['logM'].item(),
        'logRs_halo': jnp.log10(gt_params['halo_params']['scaleRadius']).item(),
        'logM_disk': gt_params['disc_params']['logM'].item(),
        'logRs_disk': jnp.log10(gt_params['disc_params']['scaleRadius']).item(),
        'logHs_disk': jnp.log10(gt_params['disc_params']['scaleHeight']).item(),
}

def log_prior(params):
  lp = 0
  return lp

def log_prob(theta):
    params = theta
    lp = log_prior(params)
    if not np.isfinite(lp):
        return -np.inf
    ll = float(logl_angular_input(params, dict_data, dict_data['total_bins']))  # convert from JAX array
    if not np.isfinite(ll):
        return -np.inf
    return lp + ll


n_grid = 1024
param_grid = pd.read_csv(path + '/quasi_random_samples_8D_unity.csv').to_numpy()
index = np.random.choice(len(param_grid), size=n_grid, replace=False)
param_grid = param_grid[index]
param_grid[:, 0] = (param_grid[:, 0] - 0.5) * 6 + ground_truth['logM_halo']
param_grid[:, 1] = (param_grid[:, 1] - 0.5) * 6 + ground_truth['logM_disk']
param_grid[:, 2] = (param_grid[:, 2] - 0.5) * 2 + ground_truth['logRs_halo']
param_grid[:, 3] = (param_grid[:, 3] - 0.5) * 2 + ground_truth['logRs_disk']
param_grid[:, 4] = (param_grid[:, 4] - 0.5) * 2 + ground_truth['logHs_disk']
param_grid[:, 5] = (param_grid[:, 5] - 0.5) * 2 * jnp.pi
param_grid[:, 6] = (param_grid[:, 6] - 0.5) * 2 * jnp.pi
param_grid[:, 7] = (param_grid[:, 7] - 0.5) * 2 * jnp.pi

log_prob_grid = []
for i in tqdm(range(n_grid)):
  log_prob_grid.append(log_prob(param_grid[i]))
log_prob_grid = np.array(log_prob_grid)

pd.DataFrame({
    'logM_halo': param_grid[:, 0],
    'logM_disk': param_grid[:, 1],
    'logRs_halo': param_grid[:, 2],
    'logRs_disk': param_grid[:, 3],
    'logHs_disk': param_grid[:, 4],
    'alpha': param_grid[:, 5],
    'beta': param_grid[:, 6],
    'gamma': param_grid[:, 7],
    'log_prob': log_prob_grid,
}).to_csv(path + '/grid_search_result_0209.csv', index=False)

20000


100%|██████████| 1024/1024 [3:08:11<00:00, 11.03s/it]


In [2]:
dict_data = get_dict_data(path)

def log_prior(theta,):
    if (6 < theta[0] < 10) and (-1 < theta[1] < 2) and (-1 < theta[2] < 1)\
    and (0 <= theta[3] < jnp.pi) and (0 <= theta[4] < jnp.pi/2) and (0 <= theta[5] < jnp.pi):
        return 0.0  # log(1) = 0 for uniform prior
    return -np.inf  # log(0) = -inf for out-of-bounds

def log_prob(theta,):
    # print(theta)
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf

    ll = logl_density(theta, dict_data, dict_data['total_bins'])

    return ll + lp

ndim = 6
nwalkers = 16  # must be >= 2 * ndim

# Initialize walkers around ground truth
# p0 = np.array([ground_truth[k] for k in param_names])
p0 = np.array([9.2, 0.3, 0., jnp.pi/4, jnp.pi/4, jnp.pi/4])
# initial_pos = p0 + 1e-1 * np.random.randn(nwalkers, ndim)
np.random.seed(42)
initial_pos = p0 + np.random.uniform(-0.3, 0.3, (nwalkers, ndim))

sampler = emcee.EnsembleSampler(nwalkers, ndim, log_prob)
sampler.run_mcmc(initial_pos, 500, progress=True)

samples = sampler.get_chain(discard=200, flat=True)

params_bestfit = np.percentile(samples, axis=0, q=50)
logl_val = logl_density(params_bestfit, dict_data, dict_data['total_bins'])
print('Best-fit logL projection', logl_val)#

dict_data['logl_density_max'] = logl_val
logrho0_best_fit, logRs_disk_best_fit, logHs_disk_best_fit, alpha_best_fit, beta_best_fit, gamma_best_fit = params_bestfit

# dict_data['logl_density_max'] = -0.24
# logrho0_best_fit, logRs_disk_best_fit, logHs_disk_best_fit, alpha_best_fit, beta_best_fit, gamma_best_fit = (9,0.45,-0.24,0.54,0.36,1.34)
# disc_mass_tot = 10**logHs_disk_best_fit * 4 * np.pi * 10**(2*logRs_disk_best_fit) * 10**logHs_disk_best_fit  # Total mass from best-fit parameters


alpha = alpha_best_fit
beta = beta_best_fit
gamma = gamma_best_fit
ground_truth = [11.5,
                logrho0_best_fit,
                jnp.log10(19).item(),
                logRs_disk_best_fit,
                logHs_disk_best_fit,
                alpha,
                beta,
                gamma,
                0.
]
logL = logl_angular_input(ground_truth, dict_data, dict_data['total_bins'])
print(logL)

import time
start = time.time()
logL = logl_angular_input(ground_truth, dict_data, dict_data['total_bins'])
end = time.time()
print('time per logl evaluation', end - start)

prior_uniform_low =  [ground_truth[0] - 3,
                    ground_truth[1] - 3,
                    ground_truth[2]- 1,
                    ground_truth[3]- 1,
                    ground_truth[4]- 1,
                    0,
                    0,
                    0,
                    -2
                    ]
prior_uniform_high = [ground_truth[0] + 3,
                    ground_truth[1] + 3,
                    ground_truth[2]+ 1,
                    ground_truth[3]+ 1,
                    ground_truth[4]+ 1,
                    jnp.pi,
                    jnp.pi/2,
                    jnp.pi,
                    2
                    ]

def log_prior(params):
    lp = 0
    for i in range (0, 9):
        if (params[i]<=prior_uniform_low[i]) & (params[i]>=prior_uniform_high[i]):
            lp+= -jnp.inf
    return lp

def log_prob(theta):
    params = theta
    lp = log_prior(params)
    if not np.isfinite(lp):
        return -np.inf
    ll = float(logl_angular_input(params, dict_data, dict_data['total_bins']))  # convert from JAX array
    if not np.isfinite(ll):
        return -np.inf
    return lp + ll

ndim = 9
nwalkers = 18  # must be >= 2 * ndim

# Initialize walkers around ground truth
# p0 = np.array([ground_truth[k] for k in param_names])
p0 = ground_truth
# initial_pos = p0 + 1e-1 * np.random.randn(nwalkers, ndim)
initial_pos = p0 + np.random.uniform(-0.3, 0.3, (nwalkers, ndim))

sampler = emcee.EnsembleSampler(nwalkers, ndim, log_prob)
sampler.run_mcmc(initial_pos, 300, progress=True)

samples = sampler.get_chain(discard=100, flat=True)

param_names = ['logM_halo','logM_disk', 'logRs_halo', 'logRs_disk', 'logHs_disk', 'alpha', 'beta', 'gamma', 'log_light_to_mass_ratio']
import pandas as pd
pd.DataFrame(samples, columns=param_names).to_csv(path+'/test_posterior_0219.csv', index=False)

samples_raw = sampler.get_chain(discard=0, flat=False)
with open(path+'/test_posterior_WholeChain_0219.pkl', 'wb') as f:
    pickle.dump(samples_raw, f)

20000


100%|██████████| 500/500 [00:03<00:00, 129.60it/s]


Best-fit logL projection -2.9707503
-56.188576
time per logl evaluation 7.8002002239227295


  0%|          | 0/300 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/emcee/moves/red_blue.py:99: RuntimeWarning: invalid value encountered in scalar subtract
  lnpdiff = f + nlp - state.log_prob[j]
100%|██████████| 300/300 [10:37:00<00:00, 127.40s/it]


In [2]:
dict_data = get_dict_data(path)

def log_prior(theta,):
    if (6 < theta[0] < 10) and (-1 < theta[1] < 2) and (-1 < theta[2] < 1)\
    and (0 <= theta[3] < jnp.pi) and (0 <= theta[4] < jnp.pi/2) and (0 <= theta[5] < jnp.pi):
        return 0.0  # log(1) = 0 for uniform prior
    return -np.inf  # log(0) = -inf for out-of-bounds

def log_prob(theta,):
    # print(theta)
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf

    ll = logl_density(theta, dict_data, dict_data['total_bins'])

    return ll + lp

ndim = 6
nwalkers = 16  # must be >= 2 * ndim

# Initialize walkers around ground truth
# p0 = np.array([ground_truth[k] for k in param_names])
p0 = np.array([9.2, 0.3, 0., jnp.pi/4, jnp.pi/4, jnp.pi/4])
# initial_pos = p0 + 1e-1 * np.random.randn(nwalkers, ndim)
np.random.seed(42)
initial_pos = p0 + np.random.uniform(-0.3, 0.3, (nwalkers, ndim))

sampler = emcee.EnsembleSampler(nwalkers, ndim, log_prob)
sampler.run_mcmc(initial_pos, 500, progress=True)

samples = sampler.get_chain(discard=200, flat=True)

params_bestfit = np.percentile(samples, axis=0, q=50)
logl_val = logl_density(params_bestfit, dict_data, dict_data['total_bins'])
print('Best-fit logL projection', logl_val)#

dict_data['logl_density_max'] = logl_val
logrho0_best_fit, logRs_disk_best_fit, logHs_disk_best_fit, alpha_best_fit, beta_best_fit, gamma_best_fit = params_bestfit

# dict_data['logl_density_max'] = -0.24
# logrho0_best_fit, logRs_disk_best_fit, logHs_disk_best_fit, alpha_best_fit, beta_best_fit, gamma_best_fit = (9,0.45,-0.24,0.54,0.36,1.34)
# disc_mass_tot = 10**logHs_disk_best_fit * 4 * np.pi * 10**(2*logRs_disk_best_fit) * 10**logHs_disk_best_fit  # Total mass from best-fit parameters


alpha = alpha_best_fit
beta = beta_best_fit
gamma = gamma_best_fit
ground_truth = [11.5,
                logrho0_best_fit,
                jnp.log10(19).item(),
                logRs_disk_best_fit,
                logHs_disk_best_fit,
                alpha,
                beta,
                gamma,
                0.
]
logL = logl_angular_input(ground_truth, dict_data, dict_data['total_bins'])
print(logL)

import time
start = time.time()
logL = logl_angular_input(ground_truth, dict_data, dict_data['total_bins'])
end = time.time()
print('time per logl evaluation', end - start)

prior_uniform_low =  [ground_truth[0] - 3,
                    ground_truth[1] - 3,
                    ground_truth[2]- 1,
                    ground_truth[3]- 1,
                    ground_truth[4]- 1,
                    0,
                    0,
                    0,
                    -2
                    ]
prior_uniform_high = [ground_truth[0] + 3,
                    ground_truth[1] + 3,
                    ground_truth[2]+ 1,
                    ground_truth[3]+ 1,
                    ground_truth[4]+ 1,
                    jnp.pi,
                    jnp.pi/2,
                    jnp.pi,
                    2
                    ]

def log_prior(params):
    lp = 0
    for i in range (0, 9):
        if (params[i]<=prior_uniform_low[i]) & (params[i]>=prior_uniform_high[i]):
            lp+= -jnp.inf
    return lp

def log_prob(theta):
    params = theta
    lp = log_prior(params)
    if not np.isfinite(lp):
        return -np.inf
    ll = float(logl_angular_input(params, dict_data, dict_data['total_bins']))  # convert from JAX array
    if not np.isfinite(ll):
        return -np.inf
    return lp + ll


n_grid = 1024
param_grid = pd.read_csv(path + '/quasi_random_samples_12D_unity.csv').to_numpy()
index = np.random.choice(len(param_grid), size=n_grid, replace=False)
param_grid = param_grid[index]
param_grid[:, 0] = (param_grid[:, 0] - 0.5) * 6 + ground_truth[0]
param_grid[:, 1] = (param_grid[:, 1] - 0.5) * 6 + ground_truth[1]
param_grid[:, 2] = (param_grid[:, 2] - 0.5) * 2 + ground_truth[2]
param_grid[:, 3] = (param_grid[:, 3] - 0.5) * 2 + ground_truth[3]
param_grid[:, 4] = (param_grid[:, 4] - 0.5) * 2 + ground_truth[4]
param_grid[:, 5] = (param_grid[:, 5] - 0.5) * 1 * jnp.pi
param_grid[:, 6] = (param_grid[:, 6] - 0.5) * 1/2 * jnp.pi
param_grid[:, 7] = (param_grid[:, 7] - 0.5) * 1 * jnp.pi
param_grid[:, 8] = (param_grid[:, 8] - 0.5) * 2

from tqdm import tqdm
log_prob_grid = []
for i in tqdm(range(n_grid)):
  log_prob_grid.append(log_prob(param_grid[i]))
log_prob_grid = np.array(log_prob_grid)

pd.DataFrame({
    'logM_halo': param_grid[:, 0],
    'logM_disk': param_grid[:, 1],
    'logRs_halo': param_grid[:, 2],
    'logRs_disk': param_grid[:, 3],
    'logHs_disk': param_grid[:, 4],
    'alpha': param_grid[:, 5],
    'beta': param_grid[:, 6],
    'gamma': param_grid[:, 7],
    'log_light_to_mass_ratio': param_grid[:, 8],
    'log_prob': log_prob_grid,
}).to_csv(path + '/grid_search_result_0219.csv', index=False)

20000


100%|██████████| 500/500 [00:03<00:00, 135.79it/s]


Best-fit logL projection -0.37391293
-3787.6433
time per logl evaluation 6.565016031265259


100%|██████████| 1024/1024 [1:00:40<00:00,  3.55s/it]


In [3]:
from google.colab import runtime
runtime.unassign()

# Initialisation (always run this)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ! pip install numpyro
# # ! pip install jax_cosmo
# ! pip install arviz
! pip install jaxopt
! pip install corner
! pip install emcee

path = '/content/drive/MyDrive/SchwarMAX-MCMC/'

import sys
sys.path.append(path)

from model import *
from likelihoods import *
from utils import *
from sample_from_density import sample_from_density_grid
from CylindricalSpline import get_phi_m, evaluate_phi_axisymmetric

import jax
import jax.numpy as jnp
import jax.numpy.linalg as jnn
import pandas as pd
import numpy as np
import scipy as sp
import pickle

import emcee
import corner
import matplotlib.pyplot as plt

from constants import EPSILON

def get_dict_data(path):

    with open(path + 'mock_Nbody_disc_bulge_XY_withRot.pkl', 'rb') as f:
        bin_dict = pickle.load(f)

    # voronoi binning mapping and data
    num_per_bin = jnp.array(bin_dict['num_per_bin'])
    total_bins = jnp.array(bin_dict['total_bins'])
    bin_mapping = jnp.array(bin_dict['bin_mapping'])
    surface_density = jnp.array(bin_dict['surface_density'])
    V_data = jnp.array(bin_dict['V_mean'])
    sigma_data = jnp.array(bin_dict['V_sigma'])
    h1_data = jnp.array(bin_dict['h1'])
    h2_data = jnp.array(bin_dict['h2'])
    h3_data = jnp.array(bin_dict['h3'])
    h4_data = jnp.array(bin_dict['h4'])
    v0 = jnp.array(bin_dict['v0'])
    s = jnp.array(bin_dict['s'])
    alpha, beta, gamma = bin_dict['orientation']

    V_data_err = jnp.where(0.1 * jnp.fabs(V_data) < 10, 10, 0.1 * V_data)
    sigma_data_err = jnp.where(0.1 * jnp.fabs(sigma_data) < 5, 5, 0.1 * sigma_data)
    h1_data_err = jnp.where(0.1 * jnp.fabs(h1_data) < 0.03, 0.03, 0.1 * jnp.fabs(h1_data))
    h2_data_err = jnp.where(0.1 * jnp.fabs(h2_data) < 0.03, 0.03, 0.1 * jnp.fabs(h2_data))
    h3_data_err = jnp.where(0.1 * jnp.fabs(h3_data) < 0.03, 0.03, 0.1 * jnp.fabs(h3_data))
    h4_data_err = jnp.where(0.1 * jnp.fabs(h4_data) < 0.03, 0.03, 0.1 * jnp.fabs(h4_data))

    # df_Rzphi_data = pd.read_csv(path + 'mock_axisymmetric_disc_Rzphi.csv')
    # Rzphi_density_data = jnp.array(df_Rzphi_data['mass'].to_numpy()).astype(jnp.float32)
    with open(path + 'mock_axisymmetric_disc_Rzphi.pkl', 'rb') as f:
        Rzphi_density_data = pickle.load(f)

    R_grid, z_grid, phi_grid = Rzphi_density_data['R_grid'], Rzphi_density_data['z_grid'], Rzphi_density_data['phi_grid']
    dR = np.unique(R_grid)[1] - np.unique(R_grid)[0]
    dz = np.unique(z_grid)[1] - np.unique(z_grid)[0]
    dphi = np.unique(phi_grid)[1] - np.unique(phi_grid)[0]
    sample_for_integration = Rzphi_density_data['sample_for_integration']

    from scipy.stats import qmc
    X_regular_grid, Y_regular_grid = bin_dict['X_regular_grid'], bin_dict['Y_regular_grid']
    dX = jnp.unique(X_regular_grid)[1] - jnp.unique(X_regular_grid)[0]
    dY = jnp.unique(Y_regular_grid)[1] - jnp.unique(Y_regular_grid)[0]
    sampler = qmc.Sobol(d=3, scramble=False)
    sample = sampler.random_base2(m=10)


    dict_data = {
        # 'w0': w0,
        'v0': v0,
        's': s,

        # 'Rzphi_density_data': Rzphi_density_data,
        'XY_density_data': surface_density,
        'V_data': V_data,
        'V_data_err': V_data_err,
        'sigma_data': sigma_data,
        'sigma_data_err': sigma_data_err,
        'h1_data': h1_data,
        'h1_data_err': h1_data_err,
        'h2_data': h2_data,
        'h2_data_err': h2_data_err,
        'h3_data': h3_data,
        'h3_data_err': h3_data_err,
        'h4_data': h4_data,
        'h4_data_err': h4_data_err,
        'num_per_bin': num_per_bin,
        'bin_mapping': bin_mapping,
        'total_bins': total_bins.item(),

        'R_grid': R_grid,
        'z_grid': z_grid,
        'phi_grid': phi_grid,
        'dR': dR,
        'dz': dz,
        'dphi': dphi,
        'sample_for_integration': sample_for_integration,

        'X_regular_grid': X_regular_grid,
        'Y_regular_grid': Y_regular_grid,
        'dX': dX,
        'dY': dY,
        'sample_for_integration_XY': sample,
    }

    return dict_data

# Load data and preprocesses (always run this)

In [ ]:
dict_data = get_dict_data(path)

def log_prior(theta,):
    if (6 < theta[0] < 10) and (8 < theta[1] < 12) and (-1 < theta[2] < 2) and (-1 < theta[3] < 1) and (-1 < theta[4] < 1)\
    and (0 <= theta[5] < jnp.pi) and (0 <= theta[6] < jnp.pi/2) and (0 <= theta[7] < jnp.pi):
        return 0.0  # log(1) = 0 for uniform prior
    return -np.inf  # log(0) = -inf for out-of-bounds

def log_prob(theta,):
    # print(theta)
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    
    ll = logl_density(theta, dict_data, dict_data['total_bins'])

    return ll + lp

ndim = 8
nwalkers = 16  # must be >= 2 * ndim

# Initialize walkers around ground truth
# p0 = np.array([ground_truth[k] for k in param_names])
p0 = np.array([9.2, 10, 0.3, 0., 0., jnp.pi/4, jnp.pi/4, jnp.pi/4])
# initial_pos = p0 + 1e-1 * np.random.randn(nwalkers, ndim)
np.random.seed(42)
initial_pos = p0 + np.random.uniform(-0.3, 0.3, (nwalkers, ndim))

sampler = emcee.EnsembleSampler(nwalkers, ndim, log_prob)
sampler.run_mcmc(initial_pos, 500, progress=True)

samples = sampler.get_chain(discard=200, flat=True)


params_bestfit = np.percentile(samples, axis=0, q=50)
logl_val = logl_density(params_bestfit, dict_data, dict_data['total_bins'])
print('Best-fit logL projection', logl_val)#

dict_data['logl_density_max'] = logl_val

logrho0_best_fit, logM_bulge_best_fit, \
logRd_disc_best_fit, logHs_disk_best_fit, logRs_bulge_best_fit, \
alpha_best_fit, beta_best_fit, gamma_best_fit = params_bestfit

params_halo_pot = {
    'logM': 11.8,
    'Rs':19.0,
    'a':1.0,
    'b':1.0,
    'c':1.0,
    'x_origin':0.0,
    'y_origin':0.0,
    'z_origin':0.0,
    'dirx':0.0,
    'diry':0.0,
    'dirz':1.0
}

params_disk_rho = {
    'rho0_disc': 10 ** logrho0_best_fit,
    'Rd_disc': 10 ** logRd_disc_best_fit,
    'hz_disc': 10 ** logHs_disk_best_fit,
    'x_origin': 0.0,
    'y_origin': 0.0,
    'z_origin': 0.0,
    'dirx': 0.0,
    'diry': 0.0,
    'dirz': 1.0,
    'alpha': alpha_best_fit * 180 / jnp.pi,
    'beta': beta_best_fit * 180 / jnp.pi,
    'gamma': gamma_best_fit * 180 / jnp.pi,
    'light_to_mass_ratio': 1,
    'logM_bulge': logM_bulge_best_fit,
    'Rs_bulge': 10 ** logRs_bulge_best_fit,
}

@jax.jit
def potential_func(x, y, z, dict_phi, params_halo):
    """ Returns Phi(R, z) """
    phi_halo = NFW_potential(x, y, z, params_halo)
    phi_disk = evaluate_phi_axisymmetric(x, y, z, dict_phi)
    return phi_halo + phi_disk

@jax.jit
def density_func(x, y, z, params):
    """ Returns Stellar Density nu(R, z) """
    # Double Exponential Disk
    val = DoubleExponentialDisk_density(x, y, z, params) + Hernquist_density(x, y, z, params)
    return val

NR, NZ, Rmin, Rmax, Zmin, Zmax, Mmax = 50, 30, 1e-2, 30.0, 1e-3, 15.0, 8.
Nphi = 200
N_int = 10_000
dict_phi = get_phi_m(density_func, params_disk_rho, NR, NZ, Rmin, Rmax, Zmin, Zmax, Mmax, Nphi, N_int)

bounds = jnp.array(
    [
        [-15.0, 15.0],  # x
        [-15.0, 15.0],  # y
        [-5.0, 5.0],    # z
    ],
    dtype=jnp.float32,
)

n_samples = 20_000
n_x,n_y,n_z = 48, 48, 32

key = jax.random.PRNGKey(20262)
sample_ic_dict = sample_from_density_grid(
    key,
    density_func,
    params_disk_rho,
    bounds,
    n_samples=n_samples,
    n_x=n_x,
    n_y=n_y,
    n_z=n_z,
)
samples = np.asarray(sample_ic_dict["samples"])

_R = jnp.sqrt(samples[:,0]**2 + samples[:,1]**2)
_z = samples[:,2]

T_orb = jax.vmap(estimate_orbital_timescale, in_axes=(0, None, None, 0))(
    _R,
    potential_func,
    (dict_phi, params_halo_pot),
    _z
)

mask = T_orb<40/1e3
w0_hres = samples[mask]
w0_lres = samples[~mask]

dict_data['w0_lres'] = jnp.array(w0_lres)
dict_data['w0_hres'] = jnp.array(w0_hres)

print("High res sample size: ", w0_hres.shape[0], "Low res sample size: ", w0_lres.shape[0])

# MCMC

In [ ]:
ground_truth = [
    11.5,
    logrho0_best_fit,
    logM_bulge_best_fit,
    jnp.log10(19).item(),
    logRs_disk_best_fit,
    logHs_disk_best_fit,
    logRs_bulge_best_fit,
    alpha_best_fit,
    beta_best_fit,
    gamma_best_fit,
    0.3
]
logL = logl_angular_input(ground_truth, dict_data, dict_data['total_bins'])
print(logL)

import time
start = time.time()
logL = logl_angular_input(ground_truth, dict_data, dict_data['total_bins'])
end = time.time()
print('time per logl evaluation', end - start)

prior_uniform_low =  [
    ground_truth[0] - 3,
    ground_truth[1] - 3,
    ground_truth[2] - 3,
    ground_truth[3]- 1,
    ground_truth[4]- 1,
    ground_truth[5]- 1,
    ground_truth[6]- 1,
    0,
    0,
    0,
    -2
]
prior_uniform_high = [
    ground_truth[0] + 3,
    ground_truth[1] + 3,
    ground_truth[2] + 3,
    ground_truth[3]+ 1,
    ground_truth[4]+ 1,
    ground_truth[5]+ 1,
    ground_truth[6]+ 1,
    jnp.pi,
    jnp.pi/2,
    jnp.pi,
    2
]

def log_prior(params):
    lp = 0
    for i in range (0, ndim):
        if (params[i]<=prior_uniform_low[i]) & (params[i]>=prior_uniform_high[i]):
            lp+= -jnp.inf
    return lp

def log_prob(theta):
    params = theta
    lp = log_prior(params)
    if not np.isfinite(lp):
        return -np.inf
    ll = float(logl_angular_input(params, dict_data, dict_data['total_bins']))  # convert from JAX array
    if not np.isfinite(ll):
        return -np.inf
    return lp + ll

ndim = 11
nwalkers = 22  # must be >= 2 * ndim

np.random.seed(42)

# Initialize walkers around ground truth
# p0 = np.array([ground_truth[k] for k in param_names])
p0 = ground_truth
initial_pos = np.zeros((nwalkers, ndim))
initial_pos[:7, :] = p0[:7] + np.random.uniform(-0.5, 0.5, (7, ndim))
initial_pos[7:10, :] = p0[7:10] + np.random.uniform(-0.1, 0.1, (3, ndim))
initial_pos[7:10, :] = np.clip(initial_pos[7:10, :], a_min=0, a_max=[jnp.pi, jnp.pi/2, jnp.pi])
initial_pos[10, :] = p0[10] + np.random.uniform(-0.3, 0.3, ndim)

sampler = emcee.EnsembleSampler(nwalkers, ndim, log_prob)
sampler.run_mcmc(initial_pos, 250, progress=True)

samples = sampler.get_chain(discard=80, flat=True)

param_names = ['logM_halo','logM_disk','logM_bulge', 'logRs_halo', 'logRs_disk', 'logHs_disk', 'logRs_bulge', 'alpha', 'beta', 'gamma', 'log_light_to_mass_ratio']
import pandas as pd
pd.DataFrame(samples, columns=param_names).to_csv(path+'/test_posterior_0225.csv', index=False)

samples_raw = sampler.get_chain(discard=0, flat=False)
with open(path+'/test_posterior_WholeChain_0225.pkl', 'wb') as f:
    pickle.dump(samples_raw, f)

# Grid search

In [ ]:
def log_prior(params):
    lp = 0
    for i in range (0, ndim):
        if (params[i]<=prior_uniform_low[i]) & (params[i]>=prior_uniform_high[i]):
            lp+= -jnp.inf
    return lp

def log_prob(theta):
    params = theta
    lp = log_prior(params)
    if not np.isfinite(lp):
        return -np.inf
    ll = float(logl_angular_input(params, dict_data, dict_data['total_bins']))  # convert from JAX array
    if not np.isfinite(ll):
        return -np.inf
    return lp + ll

ground_truth = [
    11.5,
    logrho0_best_fit,
    logM_bulge_best_fit,
    jnp.log10(19).item(),
    logRs_disk_best_fit,
    logHs_disk_best_fit,
    logRs_bulge_best_fit,
    alpha_best_fit,
    beta_best_fit,
    gamma_best_fit,
    0.
]
prior_uniform_low =  [
    ground_truth[0] - 3,
    ground_truth[1] - 3,
    ground_truth[2] - 3,
    ground_truth[3]- 1,
    ground_truth[4]- 1,
    ground_truth[5]- 1,
    ground_truth[6]- 1,
    0,
    0,
    0,
    -2
]
prior_uniform_high = [
    ground_truth[0] + 3,
    ground_truth[1] + 3,
    ground_truth[2] + 3,
    ground_truth[3]+ 1,
    ground_truth[4]+ 1,
    ground_truth[5]+ 1,
    ground_truth[6]+ 1,
    jnp.pi,
    jnp.pi/2,
    jnp.pi,
    2
]

ndim = 11

n_grid = 1024
param_grid = pd.read_csv(path + '/quasi_random_samples_12D_unity.csv').to_numpy()
index = np.random.choice(len(param_grid), size=n_grid, replace=False)
param_grid = param_grid[index]
param_grid[:, 0] = (param_grid[:, 0] - 0.5) * 6 + ground_truth[0]
param_grid[:, 1] = (param_grid[:, 1] - 0.5) * 6 + ground_truth[1]
param_grid[:, 2] = (param_grid[:, 2] - 0.5) * 6 + ground_truth[2]
param_grid[:, 2] = (param_grid[:, 3] - 0.5) * 2 + ground_truth[3]
param_grid[:, 3] = (param_grid[:, 4] - 0.5) * 2 + ground_truth[4]
param_grid[:, 4] = (param_grid[:, 5] - 0.5) * 2 + ground_truth[5]
param_grid[:, 4] = (param_grid[:, 6] - 0.5) * 2 + ground_truth[6]
param_grid[:, 5] = (param_grid[:, 7] - 0.5) * 0.1 * jnp.pi + ground_truth[7]
param_grid[:, 6] = (param_grid[:, 8] - 0.5) * 0.1 * jnp.pi + ground_truth[8]
param_grid[:, 7] = (param_grid[:, 9] - 0.5) * 0.1 * jnp.pi + ground_truth[9]
param_grid[:, 8] = (param_grid[:, 10] - 0.5) * 2

from tqdm import tqdm
log_prob_grid = []
for i in tqdm(range(n_grid)):
  log_prob_grid.append(log_prob(param_grid[i]))
log_prob_grid = np.array(log_prob_grid)

pd.DataFrame({
    'logM_halo': param_grid[:, 0],
    'logM_disk': param_grid[:, 1],
    'logM_bulge': param_grid[:, 2],
    'logRs_halo': param_grid[:, 3],
    'logRs_disk': param_grid[:, 4],
    'logHs_disk': param_grid[:, 5],
    'logRs_bulge': param_grid[:, 6],
    'alpha': param_grid[:, 7],
    'beta': param_grid[:, 8],
    'gamma': param_grid[:, 9],
    'log_light_to_mass_ratio': param_grid[:, 10],
    'log_prob': log_prob_grid,
}).to_csv(path + '/grid_search_result_0225.csv', index=False)